In [ ]:
import sys

from pathlib import Path

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

from config_loaders.dataset_config_loader import DatasetConfig
from datasets import load_dataset, load_from_disk
from utils.data_io import make_dir, save_jsonl
from utils.question_formatter import QuestionFormatter

In [2]:
ds_config = DatasetConfig()
ds_paths = ds_config.get_ds_paths("vqav2_holdout")
ds_paths

{'is_the_person': {'imgs': 'data/datasets/vqav2/holdout/imgs/is_the_person',
  'questions': 'data/datasets/vqav2/holdout/q_is_the_person.jsonl'},
 'is_this_person': {'imgs': 'data/datasets/vqav2/holdout/imgs/is_this_person',
  'questions': 'data/datasets/vqav2/holdout/q_is_this_person.jsonl'}}

In [3]:
question_formatter = QuestionFormatter("vqav2_holdout")

### Downloading and extracting data (takes time)

Skip to next section if arrow files are already present

In [4]:
ds = load_dataset("lmms-lab/VQAv2", split="validation")

Using the latest cached version of the dataset since lmms-lab/VQAv2 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/lorenzocorti/.cache/huggingface/datasets/lmms-lab___vq_av2/default/0.0.0/32665d35052eb4a6d4414851c3c829a72754915a (last modified on Thu Feb  6 11:40:33 2025).


Loading dataset shards:   0%|          | 0/67 [00:00<?, ?it/s]

In [5]:
ds

Dataset({
    features: ['question_type', 'multiple_choice_answer', 'answers', 'image_id', 'answer_type', 'question_id', 'question', 'image'],
    num_rows: 214354
})

In [6]:
ds[0]

{'question_type': 'none of the above',
 'multiple_choice_answer': 'down',
 'answers': [{'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 1},
  {'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 2},
  {'answer': 'at table', 'answer_confidence': 'yes', 'answer_id': 3},
  {'answer': 'skateboard', 'answer_confidence': 'yes', 'answer_id': 4},
  {'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 5},
  {'answer': 'table', 'answer_confidence': 'yes', 'answer_id': 6},
  {'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 7},
  {'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 8},
  {'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 9},
  {'answer': 'down', 'answer_confidence': 'yes', 'answer_id': 10}],
 'image_id': 262148,
 'answer_type': 'other',
 'question_id': 262148000,
 'question': 'Where is he looking?',
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x512>}

In [7]:
cats_to_keep = [
    "is the person",
    "is this person",
]

Filter and save "Is the person..."

In [8]:
vqa_ithep = ds.filter(
    lambda entry: entry["question_type"] == "is the person", num_proc=4
)

Filter (num_proc=4):   0%|          | 0/214354 [00:00<?, ? examples/s]

In [9]:
backup_path = "../data/datasets/vqav2/data_backup/is_the_person"
vqa_ithep.save_to_disk(backup_path)

Saving the dataset (0/1 shards):   0%|          | 0/794 [00:00<?, ? examples/s]

Filter and save "Is this person..."

In [10]:
vqa_ithisp = ds.filter(
    lambda entry: entry["question_type"] == "is this person", num_proc=4
)

Filter (num_proc=4):   0%|          | 0/214354 [00:00<?, ? examples/s]

In [11]:
backup_path = "../data/datasets/vqav2/data_backup/is_this_person"
vqa_ithisp.save_to_disk(backup_path)

Saving the dataset (0/1 shards):   0%|          | 0/734 [00:00<?, ? examples/s]

### Process "Is the person..."

In [4]:
c = "is_the_person"
backup_path = f"../data/datasets/vqav2/data_backup/{c}"
vqa_ithep = load_from_disk(backup_path)
vqa_ithep

Dataset({
    features: ['question_type', 'multiple_choice_answer', 'answers', 'image_id', 'answer_type', 'question_id', 'question', 'image'],
    num_rows: 794
})

In [5]:
q_ithep = []

img_path_col_isthep = [""] * len(vqa_ithep)
if "img_path" in vqa_ithep:
    vqa_ithep = vqa_ithep.remove_columns("img_path")

vqa_ithep = vqa_ithep.add_column("img_path", img_path_col_isthep)

out_dir = f"../{ds_paths[c]['imgs']}"
make_dir(out_dir)

for entry in vqa_ithep:
    img_path = f"{entry['image_id']}.jpg"
    entry["img_path"] = f"{out_dir}/{img_path}"

    curr_img = entry["image"].convert("RGB")
    curr_img.save(entry["img_path"])
    entry.pop("image")

    q_ithep.append(entry)

[Errno 17] File exists: '../data/datasets/vqav2/holdout/imgs/is_the_person'


In [6]:
new_q_ithep = [question_formatter.format(raw_q, c) for raw_q in q_ithep]
save_jsonl(new_q_ithep, f"../{ds_paths[c]['questions']}")

### Process "Is this person..."

In [7]:
c = "is_this_person"
backup_path = f"../data/datasets/vqav2/data_backup/{c}"
vqa_ithisp = load_from_disk(backup_path)
vqa_ithisp

Dataset({
    features: ['question_type', 'multiple_choice_answer', 'answers', 'image_id', 'answer_type', 'question_id', 'question', 'image'],
    num_rows: 734
})

In [8]:
q_ithisp = []

img_path_col_ithisp = [""] * len(vqa_ithisp)
if "img_path" in vqa_ithisp:
    vqa_ithisp = vqa_ithisp.remove_columns("img_path")

vqa_ithisp = vqa_ithisp.add_column("img_path", img_path_col_ithisp)

out_dir = f"../{ds_paths[c]['imgs']}"
make_dir(out_dir)

for entry in vqa_ithisp:
    img_path = f"{entry['image_id']}.jpg"
    entry["img_path"] = f"{out_dir}/{img_path}"

    curr_img = entry["image"].convert("RGB")
    curr_img.save(entry["img_path"])
    entry.pop("image")

    q_ithisp.append(entry)

[Errno 17] File exists: '../data/datasets/vqav2/holdout/imgs/is_this_person'


In [9]:
new_q_ithisp = [question_formatter.format(raw_q, c) for raw_q in q_ithisp]
save_jsonl(new_q_ithisp, f"../{ds_paths[c]['questions']}")